# Pairs Trading Across Sector ETFs: Mean Reversion from Weighting & Flow Frictions

**MGMTMFE-413 Statistical Arbitrage Research Project**

## Project Overview

This notebook implements a comprehensive pairs trading strategy on same-sector ETF pairs that:
- Uses time-varying hedge ratios via Kalman filtering
- Models spread mean reversion as an OU/AR(1) process
- Conditions signals on flow differentials and holdings overlap
- Implements robust backtesting with purged & embargoed walk-forward validation
- Includes robustness checks (SPA test, structural breaks, event studies)

## Table of Contents

1. [Setup and Configuration](#setup)
2. [Data Loading and Preprocessing](#data)
3. [Hedge Ratio Estimation (Kalman Filter)](#kalman)
4. [Mean Reversion Analysis](#meanrev)
5. [Signal Generation](#signals)
6. [Backtesting](#backtest)
7. [Performance Evaluation](#performance)
8. [Robustness Checks](#robustness)
9. [Results and Interpretation](#results)



## 1. Setup and Configuration {#setup}


In [ ]:
import sys
from pathlib import Path

# Add src to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Import project modules
import config
from data_loader import ETFDataLoader
from kalman_filter import KalmanFilter, estimate_hedge_ratio_kalman
from mean_reversion import estimate_ar1, rolling_ar1, adf_test, compute_mean_reversion_metrics
from signal_generation import generate_signals, compute_position_sizes, compute_zscore
from backtesting import WalkForwardBacktest, run_backtest
from performance import compute_performance_metrics, compute_rolling_metrics
from robustness import (
    spa_test, detect_structural_breaks, event_study,
    identify_large_flow_events, compute_robustness_checks
)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print("✓ Imports successful")
print(f"Project root: {project_root}")
print(f"ETF Pairs: {config.ETF_PAIRS}")


## 2. Data Loading and Preprocessing {#data}

**Note:** This section assumes you have data files or will connect to WRDS/Refinitiv.
For demonstration, we'll show the structure and provide placeholder code.


In [ ]:
# Initialize data loader
loader = ETFDataLoader(data_dir='data/raw')

# Example: Load data for a pair
# In production, you would:
# 1. Connect to WRDS/Refinitiv API
# 2. Download price, volume, shares outstanding data
# 3. Download holdings snapshots

# For demonstration, we'll create synthetic data structure
# Replace this with actual data loading

print("Data loader initialized.")
print("\nExpected data structure:")
print("- Price data: Date, Close/Adj Close, Volume, Shares Outstanding")
print("- Holdings data: Date, Ticker, Weight")
print("\nTo load real data:")
print("loader.load_price_data('XLE', file_path='data/raw/XLE.csv')")
print("loader.load_holdings_data('XLE', file_path='data/raw/XLE_holdings.csv')")


## 3. Hedge Ratio Estimation (Kalman Filter) {#kalman}

Estimate time-varying hedge ratio using Kalman filter.


In [ ]:
# Example workflow for Kalman filter estimation
# Assuming pair_data is loaded

def estimate_pair_hedge_ratio(symbol_a, symbol_b, pair_data):
    """
    Estimate hedge ratio for an ETF pair.
    
    This is a template - replace with actual data loading.
    """
    # Extract log prices
    log_price_a = pair_data['log_price_A']
    log_price_b = pair_data['log_price_B']
    
    # Run Kalman filter
    kalman_result = estimate_hedge_ratio_kalman(
        pair_data,
        config=config.KALMAN_CONFIG
    )
    
    return kalman_result

# Example usage (uncomment when data is loaded):
# pair_data = loader.create_pair_dataset('XLE', 'VDE', 
#                                       start_date=config.START_DATE,
#                                       end_date=config.END_DATE)
# kalman_results = estimate_pair_hedge_ratio('XLE', 'VDE', pair_data)

print("Kalman filter estimation function ready.")
print("\nKey outputs:")
print("- beta_smoothed: Time-varying hedge ratio")
print("- spread: Spread series s_t = log_price_A - beta_t * log_price_B")


## 4. Mean Reversion Analysis {#meanrev}

Model spread as AR(1) process and compute half-life.


In [ ]:
# Example mean reversion analysis
# Assuming spread is computed from Kalman filter

def analyze_mean_reversion(spread):
    """
    Analyze mean reversion properties of spread.
    """
    # Compute comprehensive metrics
    metrics = compute_mean_reversion_metrics(
        spread,
        config=config.MEAN_REVERSION_CONFIG
    )
    
    # Rolling AR(1) estimation
    rolling_ar1_results = rolling_ar1(
        spread,
        window=config.MEAN_REVERSION_CONFIG['ar1_window']
    )
    
    return metrics, rolling_ar1_results

# Example usage:
# metrics, rolling = analyze_mean_reversion(kalman_results['spread'])

print("Mean reversion analysis functions ready.")
print("\nKey metrics:")
print("- mu: Mean reversion level")
print("- phi: AR(1) coefficient")
print("- half_life: Half-life in days")
print("- is_stationary: ADF test result")


## 5. Signal Generation {#signals}

Generate trading signals with z-scores and flow/weighting conditions.


In [ ]:
# Signal generation example

def generate_pair_signals(pair_data, spread, signal_config=None):
    """
    Generate trading signals for a pair.
    """
    if signal_config is None:
        signal_config = config.SIGNAL_CONFIG
    
    # Generate signals
    signals = generate_signals(pair_data, spread, signal_config)
    
    # Compute position sizes
    signals = compute_position_sizes(
        signals,
        pair_data,
        config.BACKTEST_CONFIG
    )
    
    return signals

# Example usage:
# signals = generate_pair_signals(pair_data, kalman_results['spread'])

print("Signal generation functions ready.")
print("\nSignal logic:")
print("- Enter long when z-score < -z_entry AND flow condition met")
print("- Enter short when z-score > +z_entry AND flow condition met")
print("- Exit when |z-score| < z_exit OR max holding period OR stop loss")


## 6. Backtesting {#backtest}

Run walk-forward backtest with purged and embargoed splits.


In [ ]:
# Backtesting example

def run_pair_backtest(pair_data, spread, signal_config, cost_config):
    """
    Run full backtest for a pair.
    """
    # Initialize walk-forward backtest
    backtest = WalkForwardBacktest(
        train_window_years=config.BACKTEST_CONFIG['train_window_years'],
        test_window_years=config.BACKTEST_CONFIG['test_window_years'],
        purge_days=config.BACKTEST_CONFIG['purge_days'],
        embargo_days=config.BACKTEST_CONFIG['embargo_days'],
        start_date=config.START_DATE,
        end_date=config.END_DATE
    )
    
    # Create folds
    folds = backtest.create_folds(pair_data)
    print(f"Created {len(folds)} walk-forward folds")
    
    # Run backtest (with parameter optimization)
    results = backtest.run_backtest(
        pair_data,
        spread,
        param_grid=config.PARAM_GRID,
        signal_config=signal_config,
        cost_config=cost_config
    )
    
    return results, folds

# Example usage:
# backtest_results, folds = run_pair_backtest(
#     pair_data,
#     kalman_results['spread'],
#     config.SIGNAL_CONFIG,
#     config.BACKTEST_CONFIG
# )

print("Backtesting framework ready.")
print("\nFeatures:")
print("- Walk-forward validation")
print("- Purged and embargoed splits")
print("- Parameter optimization on training sets")
print("- Transaction costs and capacity constraints")


## 7. Performance Evaluation {#performance}

Compute comprehensive performance metrics.


In [ ]:
# Performance evaluation example

def evaluate_performance(returns, signals):
    """
    Compute performance metrics.
    """
    metrics = compute_performance_metrics(returns, signals)
    
    # Rolling metrics
    rolling_metrics = compute_rolling_metrics(returns)
    
    return metrics, rolling_metrics

# Example usage:
# metrics, rolling = evaluate_performance(strategy_returns, signals)

print("Performance evaluation functions ready.")
print("\nMetrics computed:")
print("- Return: Total, annualized")
print("- Risk: Volatility, Sharpe, max drawdown, Calmar")
print("- Trade-level: Win rate, avg holding period, profit factor")
print("- Capacity: Turnover, notional usage")


## 8. Robustness Checks {#robustness}

SPA test, structural breaks, event studies.


In [ ]:
# Robustness checks example

def run_robustness_checks(pair_data, spread, signals, returns, beta_estimates):
    """
    Run all robustness checks.
    """
    robustness_results = compute_robustness_checks(
        pair_data,
        spread,
        signals,
        returns,
        beta_estimates,
        config=config.ROBUSTNESS_CONFIG
    )
    
    return robustness_results

# Example usage:
# robustness = run_robustness_checks(
#     pair_data,
#     kalman_results['spread'],
#     signals,
#     strategy_returns,
#     kalman_results['beta_smoothed']
# )

print("Robustness check functions ready.")
print("\nChecks included:")
print("- SPA test: Data snooping bias adjustment")
print("- Structural breaks: Parameter stability")
print("- Event studies: Behavior around large flows/rebalances")


## 9. Complete Workflow Example {#workflow}

This cell demonstrates the complete workflow for a single pair.


In [ ]:
def run_complete_analysis(symbol_a, symbol_b):
    """
    Run complete analysis pipeline for an ETF pair.
    
    This is a template function - replace data loading with actual sources.
    """
    print(f"\n{'='*60}")
    print(f"Analyzing pair: {symbol_a} - {symbol_b}")
    print(f"{'='*60}\n")
    
    # Step 1: Load data
    print("Step 1: Loading data...")
    # pair_data = loader.create_pair_dataset(symbol_a, symbol_b,
    #                                        start_date=config.START_DATE,
    #                                        end_date=config.END_DATE)
    # holdings_metrics = loader.compute_pair_holdings_metrics(symbol_a, symbol_b)
    # print(f"Holdings overlap: {holdings_metrics['overlap']:.2f}")
    # print(f"Weighting distance: {holdings_metrics['weighting_distance']:.2f}")
    
    # Step 2: Estimate hedge ratio
    print("\nStep 2: Estimating hedge ratio (Kalman filter)...")
    # kalman_results = estimate_hedge_ratio_kalman(pair_data, config.KALMAN_CONFIG)
    # print(f"Beta range: [{kalman_results['beta_smoothed'].min():.3f}, "
    #       f"{kalman_results['beta_smoothed'].max():.3f}]")
    
    # Step 3: Mean reversion analysis
    print("\nStep 3: Mean reversion analysis...")
    # metrics, rolling = analyze_mean_reversion(kalman_results['spread'])
    # print(f"Half-life: {metrics['half_life']:.1f} days")
    # print(f"Stationary: {metrics['is_stationary']}")
    
    # Step 4: Generate signals
    print("\nStep 4: Generating signals...")
    # signals = generate_pair_signals(pair_data, kalman_results['spread'])
    # print(f"Total signals: {(signals['signal'] != 0).sum()}")
    
    # Step 5: Backtest
    print("\nStep 5: Running backtest...")
    # backtest_results, folds = run_pair_backtest(
    #     pair_data, kalman_results['spread'],
    #     config.SIGNAL_CONFIG, config.BACKTEST_CONFIG
    # )
    
    # Step 6: Performance
    print("\nStep 6: Computing performance...")
    # strategy_returns = backtest_results['test_returns']  # Example
    # perf_metrics, rolling_perf = evaluate_performance(strategy_returns, signals)
    # print(f"Sharpe ratio: {perf_metrics['sharpe_ratio']:.2f}")
    # print(f"Max drawdown: {perf_metrics['max_drawdown']:.2%}")
    
    # Step 7: Robustness
    print("\nStep 7: Robustness checks...")
    # robustness = run_robustness_checks(
    #     pair_data, kalman_results['spread'], signals,
    #     strategy_returns, kalman_results['beta_smoothed']
    # )
    # print(f"SPA test p-value: {robustness['spa_test']['pvalue']:.3f}")
    
    print("\n✓ Analysis complete!")
    
    # return {
    #     'pair_data': pair_data,
    #     'kalman_results': kalman_results,
    #     'signals': signals,
    #     'backtest_results': backtest_results,
    #     'performance': perf_metrics,
    #     'robustness': robustness
    # }

# Example: Run for all pairs
print("Complete workflow function ready.")
print("\nTo run analysis:")
print("results = run_complete_analysis('XLE', 'VDE')")
